# CI vs CD Grid -- Synthetic AR(1) Experiments (v2)

Trains PatchTST in channel-independent (CI) and channel-dependent (CD) modes
over a 3x3x2 grid: C in {7, 21, 84} x rho in {0.1, 0.5, 0.9} x {CI, CD}.
Each cell is run with 3 random seeds; results are reported as mean and std.

**Known confound -- gradient steps at large C:**
CD at C=84 uses batch_size=4 (memory constraint: T4 16 GB). This gives CD
31.5x more gradient updates per epoch than CI at C=84 (batch_size=128).
This is an inherent limitation of the CD architecture on constrained hardware,
not a controlled design choice. Total steps per run are logged in the results
CSV and must be reported as a limitation in the paper.

**Linear baseline:**
A channel-independent DLinear (one Linear(seq_len, pred_len) per variate,
shared weights) is included as a sanity check. If CI or CD underperforms
DLinear, the transformer is not learning anything useful.

Results are saved after every run. The notebook is fully resumable:
completed (C, rho, mode, seed) combinations are skipped on rerun.

In [ ]:
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")


## Synthetic Dataset

In [ ]:
from typing import Literal

Split = Literal["train", "val", "test"]

_BURN_IN: int = 1000
_TRAIN_NUM: int = 8640
_VAL_NUM: int = 2880
_DENOM: int = 14400


def build_covariance(C: int, rho: float) -> np.ndarray:
    """Compound-symmetry covariance matrix. Positive definite for rho in (-1/(C-1), 1)."""
    if C < 2:
        raise ValueError(f"C must be >= 2; got {C}.")
    lower = -1.0 / (C - 1)
    if not (lower < rho < 1.0):
        raise ValueError(f"rho={rho} outside valid range ({lower:.6f}, 1.0) for C={C}.")
    return rho * np.ones((C, C), dtype=np.float64) + (1.0 - rho) * np.eye(C, dtype=np.float64)


def generate_ar1(T: int, C: int, phi: float, cov: np.ndarray, seed: int) -> np.ndarray:
    """Multivariate AR(1): x_t = phi*x_{t-1} + eps_t, eps_t ~ N(0, cov). Returns (T-_BURN_IN, C)."""
    if T <= _BURN_IN:
        raise ValueError(f"T={T} must be > _BURN_IN={_BURN_IN}.")
    rng = np.random.default_rng(seed)
    x = np.empty((T, C), dtype=np.float64)
    x[0] = rng.standard_normal(C)
    noise = rng.multivariate_normal(np.zeros(C), cov, size=T - 1)
    for t in range(1, T):
        x[t] = phi * x[t - 1] + noise[t - 1]
    return x[_BURN_IN:]


def measure_empirical_correlation(series: np.ndarray) -> float:
    """Return mean absolute off-diagonal Pearson correlation across all variate pairs."""
    corr = np.corrcoef(series.T)
    n = corr.shape[0]
    row_idx, col_idx = np.triu_indices(n, k=1)
    return float(np.abs(corr[row_idx, col_idx]).mean())


class SyntheticARDataset(Dataset):
    """Sliding-window dataset over a synthetic AR(1) series. Split: 60/20/20."""

    def __init__(
        self, C: int, rho: float, phi: float, seq_len: int, pred_len: int,
        split: Split, seed: int, total_len: int = 14400,
    ) -> None:
        if split not in ("train", "val", "test"):
            raise ValueError(f"split must be 'train', 'val', or 'test'; got '{split}'.")
        cov = build_covariance(C, rho)
        series = generate_ar1(total_len, C, phi, cov, seed)

        usable = len(series)
        train_end = usable * _TRAIN_NUM // _DENOM
        val_end   = train_end + usable * _VAL_NUM // _DENOM

        # Measure empirical correlation on train split for reporting
        self.empirical_rho = measure_empirical_correlation(series[:train_end])

        scaler = StandardScaler()
        scaler.fit(series[:train_end])
        normalized = scaler.transform(series).astype(np.float32)

        if split == "train":
            self._data = normalized[:train_end]
        elif split == "val":
            self._data = normalized[train_end:val_end]
        else:
            self._data = normalized[val_end:]

        window = seq_len + pred_len
        if len(self._data) < window:
            raise ValueError(
                f"Split '{split}' has {len(self._data)} rows but seq_len+pred_len={window}."
            )
        self.seq_len  = seq_len
        self.pred_len = pred_len

    def __len__(self) -> int:
        return len(self._data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx: int) -> tuple:
        x = self._data[idx : idx + self.seq_len]
        y = self._data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return torch.from_numpy(x), torch.from_numpy(y)


## Models

In [ ]:
# ── PatchTST ──────────────────────────────────────────────────────────────────

class PatchEmbedding(nn.Module):
    def __init__(self, patch_size: int, stride: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.stride     = stride
        self.projection = nn.Linear(patch_size, d_model)
        self.dropout    = nn.Dropout(dropout)
        self._d_model   = d_model

    def _sinusoidal_pe(self, num_patches: int, device: torch.device) -> torch.Tensor:
        position = torch.arange(num_patches, device=device).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, self._d_model, 2, device=device) * (-math.log(10000.0) / self._d_model)
        )
        pe = torch.zeros(num_patches, self._d_model, device=device)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x  = x.squeeze(-1).unfold(dimension=-1, size=self.patch_size, step=self.stride)
        x  = self.projection(x)
        return self.dropout(x + self._sinusoidal_pe(x.shape[1], x.device))


class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out
        return x + self.ff(self.norm2(x))


class TransformerEncoder(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_layers: int, dropout: float) -> None:
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, dropout) for _ in range(num_layers)])
        self.norm   = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)


class ForecastHead(nn.Module):
    def __init__(self, num_patches: int, d_model: int, pred_len: int, dropout: float) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.linear  = nn.Linear(num_patches * d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(self.dropout(x.flatten(1)))


class PatchTST(nn.Module):
    """PatchTST with CI (channel_mixing=False) and CD (channel_mixing=True) modes.

    Reference: Nie et al., "A Time Series Is Worth 64 Words", ICLR 2023.
    https://arxiv.org/abs/2211.14730
    """

    def __init__(
        self, seq_len: int, pred_len: int, num_variates: int,
        patch_size: int = 16, stride: int = 8,
        d_model: int = 64, num_heads: int = 8, num_layers: int = 3,
        dropout: float = 0.2, channel_mixing: bool = False,
    ) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads}).")
        self.num_variates   = num_variates
        self.channel_mixing = channel_mixing
        self.num_patches    = (seq_len - patch_size) // stride + 1
        self.embedding = PatchEmbedding(patch_size, stride, d_model, dropout)
        self.encoder   = TransformerEncoder(d_model, num_heads, num_layers, dropout)
        self.head      = ForecastHead(self.num_patches, d_model, pred_len, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, C = x.shape
        x = x.permute(0, 2, 1).reshape(B * C, L, 1)
        x = self.embedding(x)
        if self.channel_mixing:
            x = x.reshape(B, C * self.num_patches, -1)
            x = self.encoder(x)
            x = x.reshape(B * C, self.num_patches, -1)
        else:
            x = self.encoder(x)
        x = self.head(x)
        return x.reshape(B, C, -1).permute(0, 2, 1)


# ── DLinear baseline ──────────────────────────────────────────────────────────

class DLinear(nn.Module):
    """Channel-independent linear baseline.

    One shared Linear(seq_len, pred_len) applied independently per variate.
    Equivalent to DLinear without trend-seasonal decomposition.
    If CI or CD underperforms this baseline, the transformer is not learning
    meaningful temporal structure.

    Reference: Zeng et al., "Are Transformers Effective for Time Series Forecasting?",
    AAAI 2023. https://arxiv.org/abs/2205.13504
    """

    def __init__(self, seq_len: int, pred_len: int) -> None:
        super().__init__()
        self.linear = nn.Linear(seq_len, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, seq_len, C) -> transpose -> apply linear -> transpose back
        return self.linear(x.permute(0, 2, 1)).permute(0, 2, 1)  # (B, pred_len, C)


## Training Infrastructure

In [ ]:
class EarlyStopping:
    def __init__(self, patience: int = 10, checkpoint_path: str = "best.pt") -> None:
        self.patience        = patience
        self.checkpoint_path = checkpoint_path
        self.best_val_mse    = float("inf")
        self.counter         = 0
        self.best_epoch      = 0

    def step(self, val_mse: float, model: nn.Module, epoch: int) -> bool:
        if val_mse < self.best_val_mse:
            self.best_val_mse = val_mse
            self.counter      = 0
            self.best_epoch   = epoch
            torch.save(model.state_dict(), self.checkpoint_path)
        else:
            self.counter += 1
        return self.counter >= self.patience


def compute_metrics(pred: torch.Tensor, target: torch.Tensor) -> tuple:
    mse = torch.mean((pred - target) ** 2).item()
    mae = torch.mean(torch.abs(pred - target)).item()
    return mse, mae


def train_one_epoch(model, loader, optimizer, criterion) -> tuple:
    model.train()
    total_mse, total_mae, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        _, mae = compute_metrics(pred.detach(), y)
        b = x.size(0)
        total_mse += loss.item() * b
        total_mae += mae * b
        n += b
    return total_mse / n, total_mae / n


@torch.no_grad()
def evaluate(model, loader) -> tuple:
    model.eval()
    total_mse, total_mae, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x)
        mse, mae = compute_metrics(pred, y)
        b = x.size(0)
        total_mse += mse * b
        total_mae += mae * b
        n += b
    return total_mse / n, total_mae / n


## Grid Config

In [ ]:
RESULTS_DIR = Path("results")
CKPT_DIR    = Path("results/checkpoints")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = RESULTS_DIR / "results_grid.csv"

# d_model=64, num_heads=8: intentionally smaller than the paper's 128/16 to reduce
# training time. The CI vs CD gap is large enough to appear at this capacity.
# The paper acknowledges results may shift at full capacity.
#
# Batch size by (C, mode) -- T4 memory constraint:
#   CD encoder receives C*N tokens (N=63). Attention: B * H * (C*N)^2 bytes.
#   C=7:  CI/CD batch=128  (CD attn ~0.8 GB, H=8)
#   C=21: CI batch=128, CD batch=32   (CD attn ~1.8 GB, H=8)
#   C=84: CI batch=128, CD batch=4    (CD attn ~3.6 GB, H=8)
#
# KNOWN CONFOUND: CD at C=84 with batch=4 makes 31.5x more gradient updates per
# epoch than CI (1859 vs 59 steps/epoch). This is a hardware limitation, not a
# design choice. steps_per_epoch and total_steps are logged in results_grid.csv
# and must be reported as a limitation in the paper.
GRID_CONFIG = {
    "seq_len":       512,
    "pred_len":      96,
    "patch_size":    16,
    "stride":        8,
    "d_model":       64,
    "num_heads":     8,
    "num_layers":    3,
    "dropout":       0.2,
    "lr":            1e-4,
    "warmup_epochs": 5,
    "epochs":        50,
    "patience":      10,
    "phi":           0.8,
    "total_len":     14400,
    "batch_size_ci": 128,
    "batch_size_cd": {7: 128, 21: 32, 84: 4},
}

C_VALUES   = [7, 21, 84]
RHO_VALUES = [0.1, 0.5, 0.9]
MODES      = ["CI", "CD", "DLinear"]
SEEDS      = [42, 123, 456]

total_runs = len(C_VALUES) * len(RHO_VALUES) * len(MODES) * len(SEEDS)
print(f"Grid: {len(C_VALUES)} C x {len(RHO_VALUES)} rho x {len(MODES)} modes x {len(SEEDS)} seeds = {total_runs} runs")


## Run Function

In [ ]:
def run_cell(C: int, rho: float, mode: str, seed: int, config: dict, ckpt_dir: Path) -> dict:
    """Train one run and return a result dict.

    Args:
        C: Number of variates.
        rho: Off-diagonal correlation coefficient.
        mode: 'CI', 'CD', or 'DLinear'.
        seed: Random seed for this run.
        config: GRID_CONFIG dict.
        ckpt_dir: Directory for model checkpoints.

    Returns:
        Dict with keys: dataset, C, rho, empirical_rho, mode, pred_len,
        test_mse, test_mae, best_epoch, seed, steps_per_epoch, total_steps.
    """
    torch.cuda.empty_cache()

    torch.manual_seed(seed)
    random.seed(seed)
    # np.random.seed not set globally to avoid interfering with default_rng in dataset

    channel_mixing = mode == "CD"
    is_linear      = mode == "DLinear"
    batch_size     = (
        config["batch_size_ci"] if (is_linear or not channel_mixing)
        else config["batch_size_cd"][C]
    )
    ckpt_path = str(ckpt_dir / f"synth_C{C}_rho{rho}_{mode.lower()}_seed{seed}.pt")

    ds_kwargs = dict(C=C, rho=rho, phi=config["phi"], seq_len=config["seq_len"],
                     pred_len=config["pred_len"], seed=seed, total_len=config["total_len"])
    train_ds = SyntheticARDataset(split="train", **ds_kwargs)
    val_ds   = SyntheticARDataset(split="val",   **ds_kwargs)
    test_ds  = SyntheticARDataset(split="test",  **ds_kwargs)

    empirical_rho = round(train_ds.empirical_rho, 4)

    loader_kw    = {"num_workers": 2, "pin_memory": True}
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **loader_kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **loader_kw)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **loader_kw)

    steps_per_epoch = len(train_loader)

    if is_linear:
        model = DLinear(seq_len=config["seq_len"], pred_len=config["pred_len"]).to(DEVICE)
    else:
        model = PatchTST(
            seq_len=config["seq_len"], pred_len=config["pred_len"], num_variates=C,
            patch_size=config["patch_size"], stride=config["stride"],
            d_model=config["d_model"], num_heads=config["num_heads"],
            num_layers=config["num_layers"], dropout=config["dropout"],
            channel_mixing=channel_mixing,
        ).to(DEVICE)

    total_params  = sum(p.numel() for p in model.parameters())
    warmup_epochs = config["warmup_epochs"] if not is_linear else 0

    cd_tokens = C * model.num_patches if (not is_linear and channel_mixing) else "N/A"
    print(
        f"[C={C} rho={rho} empirical={empirical_rho} {mode} seed={seed}] "
        f"params={total_params:,} | batch={batch_size} | "
        f"steps/epoch={steps_per_epoch} | CD_tokens={cd_tokens}"
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-4)

    def lr_lambda(epoch: int) -> float:
        if warmup_epochs > 0 and epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, config["epochs"] - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    criterion  = nn.MSELoss()
    early_stop = EarlyStopping(patience=config["patience"], checkpoint_path=ckpt_path)
    t0         = time.time()

    for epoch in range(1, config["epochs"] + 1):
        train_mse, _ = train_one_epoch(model, train_loader, optimizer, criterion)
        val_mse, _   = evaluate(model, val_loader)
        scheduler.step()
        if epoch % 10 == 0 or epoch == 1:
            print(
                f"  Epoch {epoch:3d}/{config['epochs']} | "
                f"train {train_mse:.4f} | val {val_mse:.4f} | "
                f"lr {optimizer.param_groups[0]['lr']:.2e} | {time.time()-t0:.0f}s"
            )
        if early_stop.step(val_mse, model, epoch):
            print(f"  Early stop @ epoch {epoch}. Best: epoch {early_stop.best_epoch}.")
            break

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    test_mse, test_mae = evaluate(model, test_loader)
    total_steps = early_stop.best_epoch * steps_per_epoch
    print(
        f"  Test MSE: {test_mse:.4f} | MAE: {test_mae:.4f} | "
        f"Best val: {early_stop.best_val_mse:.4f} @ epoch {early_stop.best_epoch} "
        f"({total_steps} total steps)"
    )

    return {
        "dataset":       "synthetic_ar1",
        "C":             C,
        "rho":           rho,
        "empirical_rho": empirical_rho,
        "mode":          mode,
        "pred_len":      config["pred_len"],
        "test_mse":      round(test_mse, 6),
        "test_mae":      round(test_mae, 6),
        "best_epoch":    early_stop.best_epoch,
        "seed":          seed,
        "batch_size":    batch_size,
        "steps_per_epoch": steps_per_epoch,
        "total_steps":   total_steps,
    }


## Verification: C=7, rho=0.1, All Modes, seed=42

In [ ]:
# Run C=7, rho=0.1, seed=42 for all three modes before the full grid.
# Verifies: no OOM, loss decreases, empirical_rho matches target rho direction.

verification_results = []

# Load existing results to skip verification runs already done
if RESULTS_CSV.exists():
    _existing = pd.read_csv(RESULTS_CSV)
    _completed = set(zip(
        _existing["C"].astype(str),
        _existing["rho"].round(8).astype(str),
        _existing["mode"],
        _existing["seed"].astype(str),
    ))
else:
    _existing = pd.DataFrame()
    _completed = set()

print("=== Verification run: C=7, rho=0.1 ===")
for mode in MODES:
    key = ("7", "0.1", mode, "42")
    if key in _completed:
        print(f"[SKIP] C=7 rho=0.1 {mode} seed=42 already complete.")
        row = _existing[
            (_existing["C"] == 7) & (_existing["rho"].round(8) == 0.1) &
            (_existing["mode"] == mode) & (_existing["seed"] == 42)
        ].to_dict("records")[0]
        verification_results.append(row)
        continue
    print(f"\n--- {mode} ---")
    result = run_cell(C=7, rho=0.1, mode=mode, seed=42, config=GRID_CONFIG, ckpt_dir=CKPT_DIR)
    verification_results.append(result)
    torch.cuda.empty_cache()

print("\nVerification summary:")
for r in verification_results:
    print(f"  {r['mode']:8s} test_mse={r['test_mse']:.4f} empirical_rho={r['empirical_rho']:.4f} best_epoch={r['best_epoch']}")
print("\nIf all three modes completed without error, proceed to the full grid.")


## Full Grid Run

In [ ]:
# Resumability: use (C, rho, mode, seed) as the unique key.
# rho is rounded to 8 decimal places before string conversion to avoid
# float comparison fragility when reading back from CSV.
if RESULTS_CSV.exists():
    existing_df = pd.read_csv(RESULTS_CSV)
    completed = set(zip(
        existing_df["C"].astype(str),
        existing_df["rho"].round(8).astype(str),
        existing_df["mode"],
        existing_df["seed"].astype(str),
    ))
    all_results = existing_df.to_dict("records")
    print(f"Resuming: {len(completed)} runs complete, "
          f"{len(C_VALUES)*len(RHO_VALUES)*len(MODES)*len(SEEDS) - len(completed)} remaining.")
else:
    completed   = set()
    all_results = []
    print("Starting fresh grid run.")

# Fold in verification results not yet in the CSV
for r in verification_results:
    key = (str(r["C"]), str(round(r["rho"], 8)), r["mode"], str(r["seed"]))
    if key not in completed:
        all_results.append(r)
        completed.add(key)

for C in C_VALUES:
    for rho in RHO_VALUES:
        for mode in MODES:
            for seed in SEEDS:
                key = (str(C), str(round(rho, 8)), mode, str(seed))
                if key in completed:
                    print(f"[SKIP] C={C} rho={rho} {mode} seed={seed}")
                    continue

                print(f'\n{"=" * 60}')
                print(f"C={C} | rho={rho} | mode={mode} | seed={seed}")
                print(f'{"=" * 60}')
                result = run_cell(C=C, rho=rho, mode=mode, seed=seed,
                                  config=GRID_CONFIG, ckpt_dir=CKPT_DIR)
                all_results.append(result)
                completed.add(key)
                torch.cuda.empty_cache()

                # Save after every run
                pd.DataFrame(all_results).to_csv(RESULTS_CSV, index=False)

results_df = pd.DataFrame(all_results)

# Duplicate check before aggregation
dupes = results_df.groupby(["C", "rho", "mode", "seed"]).size()
dupes = dupes[dupes > 1]
if not dupes.empty:
    print(f"WARNING: duplicate rows found:\n{dupes}")
    results_df = results_df.drop_duplicates(subset=["C", "rho", "mode", "seed"], keep="last")

results_df.to_csv(RESULTS_CSV, index=False)

# ── Summary: mean +/- std across seeds ───────────────────────────────────────
agg = (
    results_df.groupby(["C", "rho", "mode"])[["test_mse", "test_mae"]]
    .agg(["mean", "std"])
    .round(6)
)
agg.columns = ["mse_mean", "mse_std", "mae_mean", "mae_std"]
agg = agg.reset_index()
agg.to_csv(RESULTS_DIR / "results_grid_agg.csv", index=False)

print("\n=== Grid Results (mean +/- std across 3 seeds) ===")
print(agg.to_string(index=False))

# ── MSE ratio heatmap (CI vs CD, mean across seeds) ──────────────────────────
print("\n--- MSE ratio (CD/CI) per (C, rho) cell ---")
patchtst_agg = agg[agg["mode"].isin(["CI", "CD"])].copy()
pivot = patchtst_agg.pivot_table(index=["C", "rho"], columns="mode", values="mse_mean")
pivot["ratio_cd_ci"] = (pivot["CD"] / pivot["CI"]).round(4)
pivot["winner"]      = np.where(pivot["CI"] < pivot["CD"], "CI", "CD")
print(pivot[["CI", "CD", "ratio_cd_ci", "winner"]].to_string())
print("\nratio > 1.0: CI wins | ratio < 1.0: CD wins")

# ── DLinear comparison ────────────────────────────────────────────────────────
print("\n--- DLinear vs best transformer per (C, rho) cell ---")
dlinear_agg = agg[agg["mode"] == "DLinear"][["C", "rho", "mse_mean"]].rename(
    columns={"mse_mean": "dlinear_mse"}
)
ci_agg  = agg[agg["mode"] == "CI"][["C", "rho", "mse_mean"]].rename(columns={"mse_mean": "ci_mse"})
cd_agg  = agg[agg["mode"] == "CD"][["C", "rho", "mse_mean"]].rename(columns={"mse_mean": "cd_mse"})
compare = dlinear_agg.merge(ci_agg, on=["C", "rho"]).merge(cd_agg, on=["C", "rho"])
compare["best_transformer"] = np.where(compare["ci_mse"] < compare["cd_mse"], "CI", "CD")
compare["best_transformer_mse"] = compare[["ci_mse", "cd_mse"]].min(axis=1)
compare["transformer_beats_linear"] = compare["best_transformer_mse"] < compare["dlinear_mse"]
print(compare[["C", "rho", "dlinear_mse", "best_transformer_mse", "best_transformer", "transformer_beats_linear"]].to_string(index=False))

# ── Empirical rho report ──────────────────────────────────────────────────────
print("\n--- Empirical vs target correlation (seed=42) ---")
seed42 = results_df[results_df["seed"] == 42][["C", "rho", "empirical_rho"]].drop_duplicates()
print(seed42.sort_values(["C", "rho"]).to_string(index=False))
print("\nAll results reported as observed. No sign assumed in advance.")


## Verify All Rows Present

In [ ]:
if not RESULTS_CSV.exists():
    raise RuntimeError(f"{RESULTS_CSV} not found.")

final_df = pd.read_csv(RESULTS_CSV)
expected = len(C_VALUES) * len(RHO_VALUES) * len(MODES) * len(SEEDS)

missing = {
    (C, rho, mode, seed)
    for C in C_VALUES for rho in RHO_VALUES for mode in MODES for seed in SEEDS
} - set(zip(final_df["C"], final_df["rho"].round(8), final_df["mode"], final_df["seed"]))

if missing:
    raise RuntimeError(f"Expected {expected} rows; got {len(final_df)}. Missing: {missing}")

print(f"All {expected} rows present in {RESULTS_CSV}.")
print(f"Aggregated results written to {RESULTS_DIR / 'results_grid_agg.csv'}.")
